# 따릉이 스테이션 별 수요도 예측 Baseline

## 환경 설정

In [3]:
# ==========================================
# 통합 라이브러리 설정 (Master Setup)
# ==========================================
import os
import sys
import re
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from datetime import datetime, date, timedelta
from concurrent.futures import ThreadPoolExecutor
import geopandas as gpd
from shapely import wkt
from IPython.display import display, HTML

# ------------------------------------------
# 데이터베이스 및 환경 설정
# ------------------------------------------
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, Column, Integer, String, Float, DateTime, Date, Text, func
from sqlalchemy.orm import Mapped, mapped_column, Session

# ------------------------------------------
# Scikit-learn 모델 및 유틸리티
# ------------------------------------------
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ------------------------------------------
# 모델링 및 튜닝 도구
# ------------------------------------------
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna

# ------------------------------------------
# 시각화 및 진행률 표시 도구
# ------------------------------------------
from tqdm.notebook import tqdm

# ==========================================
# 프로젝트 경로 설정 및 환경변수 로드
# ==========================================
# 상위 폴더(프로젝트 루트)를 모듈 검색 경로에 추가
sys.path.append(os.path.dirname(os.getcwd()))

# .env 파일 로드
load_dotenv()


# ==========================================
# 시각화 및 전역 환경 설정
# ==========================================
# 그래프에서 음수 부호(-) 깨짐 방지
plt.rcParams["axes.unicode_minus"] = False
# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 난수 시드 고정
SEED = 42
np.random.seed(SEED)

print("\n========== 데이터 분석 환경 설정 완료 ==========")
print(f"설정된 시드 값: {SEED}")
print("라이브러리 로드 완료")


========== 데이터 분석 환경 설정 완료 ==========
설정된 시드 값: 42
라이브러리 로드 완료


## 데이터베이스 연결 및 데이터 조회

In [5]:
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from functools import reduce

# ==========================================
# 환경 설정 및 DB 연결
# ==========================================
load_dotenv()

DB_USER     = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "password")
DB_HOST     = os.getenv("DB_HOST", "localhost")
DB_PORT     = os.getenv("DB_PORT", "3306")
DB_NAME     = os.getenv("DB_NAME", "seoul_bike")

DATABASE_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

# 데이터베이스 연결 확인
with engine.connect() as conn:
    if conn.execute(text("SELECT 1")).scalar() == 1:
        print("\n========== 데이터베이스 연결 성공 ==========")
    else:
        print("\n========== 데이터베이스 연결 실패 ==========")


# ==========================================
# 데이터베이스 테이블 로드 (DataFrame)
# ==========================================
print("========== 각 테이블 데이터 로드 시작 ==========")

# 환경 데이터
hourly_air_2024_df    = pd.read_sql_table("hourly_air_2024", con=engine)
hourly_precip_2024_df = pd.read_sql_table("hourly_precip_2024", con=engine)
hourly_snow_2024_df   = pd.read_sql_table("hourly_snow_2024", con=engine)
hourly_temp_2024_df   = pd.read_sql_table("hourly_temp_2024", con=engine)
rt_air_df             = pd.read_sql_table("rt_air", con=engine)
rt_weather_df         = pd.read_sql_table("rt_weather", con=engine)

# 인프라 데이터
infra_business_df     = pd.read_sql_table("infra_business", con=engine)
infra_park_df         = pd.read_sql_table("infra_park", con=engine)
infra_river_df        = pd.read_sql_table("infra_river", con=engine)
infra_school_df       = pd.read_sql_table("infra_school", con=engine)
infra_univ_df         = pd.read_sql_table("infra_univ", con=engine)
infra_subway_df         = pd.read_sql_table("infra_subway", con=engine)


# 인구 데이터
pop_flow_2024_df      = pd.read_sql_table("pop_flow_2024", con=engine)
pop_living_2024_df    = pd.read_sql_table("pop_living_2024", con=engine)

# 따릉이 및 기타 데이터
# rent_history_2024_df       = pd.read_sql_table("rent_history_2024", con=engine)
korea_holidays_df     = pd.read_sql_table("korea_holidays", con=engine)
station_loc_df        = pd.read_sql_table("station_loc", con=engine)
rt_bike_status_df     = pd.read_sql_table("rt_bike_status", con=engine)

print("========== 모든 테이블 데이터 로드 완료 ==========")


========== 데이터베이스 연결 성공 ==========
========== 각 테이블 데이터 로드 시작 ==========
========== 모든 테이블 데이터 로드 완료 ==========


## 전처리

### 따릉이 위치 데이터

In [7]:
KAKAO_REST_API_KEY = os.getenv("KAKAO_REST_API_KEY")

def get_lat_lon_from_kakao(address_1, address_2):
    """주소를 받아 카카오 API를 통해 위도(lat), 경도(lon)를 반환하는 함수 (재시도 로직 포함)"""
    addr1 = address_1 if pd.notna(address_1) else ""
    addr2 = address_2 if pd.notna(address_2) else ""

    query_full = f"{addr1} {addr2}".strip()
    query_partial = addr1.strip()

    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_REST_API_KEY}"}

    # API 호출을 담당하는 내부 함수
    def fetch_coordinates(query):
        if not query:
            return None
        try:
            response = requests.get(url, headers=headers, params={"query": query})
            response.raise_for_status()
            documents = response.json().get('documents')
            if documents:
                return float(documents[0]['y']), float(documents[0]['x'])
        except Exception as e:
            print(f"API 요청 에러 ({query}): {e}")
        return None

    # [플랜 A] 주소1 + 주소2 전체로 검색
    result = fetch_coordinates(query_full)
    if result:
        return result

    # [플랜 B] 전체 검색 실패 시, 주소1만 단독으로 재검색
    if query_partial and query_partial != query_full:
        result = fetch_coordinates(query_partial)
        if result:
            return result

    # 모두 실패 시
    return 0.0, 0.0

In [8]:
import pandas as pd

print("\n========== 전처리 결과 최종 확인 ==========")

# 1. 전체 데이터 요약 정보 확인
# 총 행 개수, 컬럼별 데이터 타입, 결측치(Null) 유무를 보여줍니다.
print("[데이터프레임 요약 정보]")
print(station_loc_df.info())
print("-" * 50)

# 2. 아직도 위도나 경도가 0.0인 데이터가 있는지 최종 점검
failed_df = station_loc_df[(station_loc_df['lat'] == 0.0) | (station_loc_df['lon'] == 0.0)]
print(f"[위경도 누락(0.0) 잔여 건수]: {len(failed_df)}건")

if not failed_df.empty:
    print("🚨 여전히 처리되지 않은 데이터가 있습니다:")
    print(failed_df[['station_id', 'address_1', 'address_2']])
else:
    print("✨ 모든 대여소의 위경도 데이터가 완벽하게 채워졌습니다!")
print("-" * 50)

# 3. 이전에 실패했던 특정 대여소(ST-2, ST-423) 핀셋 확인
# 수정한 함수가 제대로 작동해서 값을 가져왔는지 직접 확인합니다.
check_ids = ['ST-2', 'ST-423']
print("[이전 실패 건(ST-2, ST-423) 업데이트 결과 확인]")
check_result = station_loc_df[station_loc_df['station_id'].isin(check_ids)][['station_id', 'address_1', 'lat', 'lon']]
print(check_result)
print("-" * 50)

# 4. 상위 5개 행 데이터 샘플 보기
print("[최종 데이터 상위 5개 샘플]")
print(station_loc_df.head())

print("========== 확인 종료 ==========\n")


========== 전처리 결과 최종 확인 ==========
[데이터프레임 요약 정보]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 846 entries, 0 to 845
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   station_id  846 non-null    object        
 1   address_1   846 non-null    object        
 2   address_2   846 non-null    object        
 3   lat         846 non-null    float64       
 4   lon         846 non-null    float64       
 5   district    846 non-null    object        
 6   grid_x      846 non-null    int64         
 7   grid_y      846 non-null    int64         
 8   created_at  846 non-null    datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int64(2), object(4)
memory usage: 59.6+ KB
None
--------------------------------------------------
[위경도 누락(0.0) 잔여 건수]: 14건
🚨 여전히 처리되지 않은 데이터가 있습니다:
    station_id             address_1       address_2
5      ST-1066  서울특별시 강서구 내발산동 741-8  마곡수명산파크2단지 교차로
7      ST-1068  서울특별시 강

### 환경 데이터

In [ ]:
# ==========================================
# 환경 데이터 전처리 함수
# ==========================================
def preprocess_env(df, col_name):
    """
    환경 데이터의 이상치 제거, 날짜 변환 및 1시간 단위 리샘플링 수행
    """
    df = df.copy()

    # 불필요한 ID 컬럼 제거
    if 'id' in df.columns:
        df = df.drop(columns=['id'])

    # 날짜 변환 및 이상치(-9) 처리
    df['measure_date'] = pd.to_datetime(df['measure_date'])
    df.replace([-9, -9.0], np.nan, inplace=True)

    # 1시간 단위 리샘플링 후 평균값 계산
    df_resampled = (
        df.set_index('measure_date')
        .groupby('region_name')[col_name]
        .resample('1h')
        .mean()
        .reset_index()
    )

    return df_resampled


# ==========================================
# 데이터 전처리 실행
# ==========================================
print("\n========== 환경 데이터 전처리 시작 ==========")

processed_air    = preprocess_env(hourly_air_2024_df, 'pm10')
processed_temp   = preprocess_env(hourly_temp_2024_df, 'temperature')
processed_precip = preprocess_env(hourly_precip_2024_df, 'precipitation')
processed_snow   = preprocess_env(hourly_snow_2024_df, 'snowfall')

print("========== 환경 데이터 전처리 완료 ==========")


# ==========================================
# 최종 데이터 통합(Merge)
# ==========================================
print("========== 환경 데이터 통합 시작 ==========")

env_dfs = [processed_air, processed_temp, processed_precip, processed_snow]

# reduce를 활용한 외부 조인(outer join) 수행
env_master_2024_df = reduce(
    lambda left, right: pd.merge(left, right, on=['measure_date', 'region_name'], how='outer'),
    env_dfs
)

# 최종 결측치 처리
env_master_2024_df['precipitation'] = env_master_2024_df['precipitation'].fillna(0)
env_master_2024_df['snowfall'] = env_master_2024_df['snowfall'].fillna(0)

print("========== 환경 데이터 통합 종료 ==========")

In [ ]:
# ==========================================
# DB(SQL) 저장 로직
# ==========================================
table_name = 'env_master_2024'
env_master_2024_df.to_sql(
    name=table_name,
    con=engine,
    if_exists='replace',
    index=False
)
print(f"========== DB 적재 성공 ==========")

### 인구 데이터

In [ ]:
# ==========================================
# 생활인구 데이터 전처리
# ==========================================
print("\n========== 생활인구 데이터 전처리 시작 ==========")
district_map = {'11500': '강서구', '11560': '영등포구', '11440': '마포구', '11710': '송파구'}
pop_living_2024_df['district_name'] = pop_living_2024_df['adstrd_code_se'].map(district_map)

# 컬럼명 정리
pop_living_2024_df.rename(columns={'tot_lvpop_co': 'lvgpop_tot'}, inplace=True)

# 연령대별 합산
pop_living_2024_df['lvgpop_10s'] = pop_living_2024_df[['male_f10t14_lvpop_co', 'male_f15t19_lvpop_co', 'female_f10t14_lvpop_co', 'female_f15t19_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_20s'] = pop_living_2024_df[['male_f20t24_lvpop_co', 'male_f25t29_lvpop_co', 'female_f20t24_lvpop_co', 'female_f25t29_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_30s'] = pop_living_2024_df[['male_f30t34_lvpop_co', 'male_f35t39_lvpop_co', 'female_f30t34_lvpop_co', 'female_f35t39_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_40s'] = pop_living_2024_df[['male_f40t44_lvpop_co', 'male_f45t49_lvpop_co', 'female_f40t44_lvpop_co', 'female_f45t49_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_50s'] = pop_living_2024_df[['male_f50t54_lvpop_co', 'male_f55t59_lvpop_co', 'female_f50t54_lvpop_co', 'female_f55t59_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_60up'] = pop_living_2024_df[['male_f60t64_lvpop_co', 'male_f65t69_lvpop_co', 'male_f70t74_lvpop_co', 'female_f60t64_lvpop_co', 'female_f65t69_lvpop_co', 'female_f70t74_lvpop_co']].sum(axis=1)

# 병합을 위한 키(Key) 생성
pop_living_2024_df = pop_living_2024_df[['stdr_de_id', 'tmzon_pd_se', 'district_name', 'lvgpop_tot', 'lvgpop_10s', 'lvgpop_20s', 'lvgpop_30s', 'lvgpop_40s', 'lvgpop_50s', 'lvgpop_60up']].copy()
pop_living_2024_df['date_str'] = pop_living_2024_df['stdr_de_id'].astype(str)
pop_living_2024_df['hour_str'] = pop_living_2024_df['tmzon_pd_se'].astype(str).str.zfill(2)


# ==========================================
# 2024년 인구 통합 마스터 생성
# ==========================================
print("========== 인구 통합 마스터 생성 시작 ==========")
date_rng = pd.date_range(start='2024-01-01 00:00:00', end='2024-12-31 23:00:00', freq='h')
districts = ['강서구', '영등포구', '마포구', '송파구']

pop_master_2024_list = []
for dist in districts:
    df_temp = pd.DataFrame(date_rng, columns=['datetime'])
    df_temp['district_name'] = dist
    df_temp['date_str'] = df_temp['datetime'].dt.strftime('%Y%m%d')
    df_temp['hour_str'] = df_temp['datetime'].dt.strftime('%H')
    df_temp['weekday'] = df_temp['datetime'].dt.weekday
    df_temp['quarter_str'] = '2024' + df_temp['datetime'].dt.quarter.astype(str)
    pop_master_2024_list.append(df_temp)

pop_master_2024_df = pd.concat(pop_master_2024_list, ignore_index=True)


# ==========================================
# 유동인구 분배 및 병합 로직
# ==========================================
print("========== 유동인구 데이터 병합 및 계산 시작 ==========")
pop_flow_2024_df = pd.merge(
    pop_master_2024_df,
    pop_flow_2024_df,
    left_on=['quarter_str', 'district_name'],
    right_on=['stdr_yyqu_cd', 'signgu_cd_nm'],
    how='left'
)

# 분배 기준값 설정
time_divisors = {
    '00': 6, '01': 6, '02': 6, '03': 6, '04': 6, '05': 6,
    '06': 5, '07': 5, '08': 5, '09': 5, '10': 5,
    '11': 3, '12': 3, '13': 3, '14': 3, '15': 3, '16': 3,
    '17': 4, '18': 4, '19': 4, '20': 4,
    '21': 3, '22': 3, '23': 3
}
weekday_col_map = {
    0: 'mon_flpop_co', 1: 'tues_flpop_co', 2: 'wed_flpop_co',
    3: 'thur_flpop_co', 4: 'fri_flpop_co', 5: 'sat_flpop_co', 6: 'sun_flpop_co'
}

def calc_flow_pop(row):
    """시간대별 유동인구 분배 계산 함수"""
    h, wd = row['hour_str'], row['weekday']
    div = time_divisors.get(h, 1)

    # 시간대별 카테고리 매핑
    if h in ['00','01','02','03','04','05']: time_pop = row['tmzon_00_06_flpop_co']
    elif h in ['06','07','08','09','10']: time_pop = row['tmzon_06_11_flpop_co']
    elif h in ['11','12','13']: time_pop = row['tmzon_11_14_flpop_co']
    elif h in ['14','15','16']: time_pop = row['tmzon_14_17_flpop_co']
    elif h in ['17','18','19','20']: time_pop = row['tmzon_17_21_flpop_co']
    else: time_pop = row['tmzon_21_24_flpop_co']

    day_pop = row[weekday_col_map[wd]]
    tot_pop = row['tot_flpop_co']

    if pd.notna(tot_pop) and tot_pop > 0:
        est_tot_flwpop = (time_pop / div) * (day_pop / tot_pop) / 13
        return pd.Series([
            est_tot_flwpop,
            est_tot_flwpop * (row['agrde_10_flpop_co'] / tot_pop),
            est_tot_flwpop * (row['agrde_20_flpop_co'] / tot_pop),
            est_tot_flwpop * (row['agrde_30_flpop_co'] / tot_pop),
            est_tot_flwpop * (row['agrde_40_flpop_co'] / tot_pop),
            est_tot_flwpop * (row['agrde_50_flpop_co'] / tot_pop),
            est_tot_flwpop * (row['agrde_60_above_flpop_co'] / tot_pop)
        ])
    return pd.Series([0, 0, 0, 0, 0, 0, 0])

# 유동인구 계산 적용
pop_flow_2024_df[['flwpop_tot', 'flwpop_10s', 'flwpop_20s', 'flwpop_30s', 'flwpop_40s', 'flwpop_50s', 'flwpop_60up']] = \
    pop_flow_2024_df.apply(calc_flow_pop, axis=1)


# ==========================================
# 최종 병합 및 정리
# ==========================================
print("========== 최종 데이터 병합 완료 ==========")
pop_master_2024_df = pd.merge(
    pop_flow_2024_df,
    pop_living_2024_df,
    on=['date_str', 'hour_str', 'district_name'],
    how='left'
)

# 핵심 컬럼만 추출
pop_master_2024_cols = [
    'datetime', 'district_name',
    'flwpop_tot', 'flwpop_10s', 'flwpop_20s', 'flwpop_30s', 'flwpop_40s', 'flwpop_50s', 'flwpop_60up',
    'lvgpop_tot', 'lvgpop_10s', 'lvgpop_20s', 'lvgpop_30s', 'lvgpop_40s', 'lvgpop_50s', 'lvgpop_60up'
]
pop_master_2024_df = pop_master_2024_df[pop_master_2024_cols]

In [ ]:
# ==========================================
# DB(SQL) 저장 로직
# ==========================================
table_name = 'pop_master_2024'
pop_master_2024_df.to_sql(
    name=table_name,
    con=engine,
    if_exists='replace',
    index=False
)
print(f"========== DB 적재 성공 ==========")

### 인프라 데이터

In [ ]:
# 각 인프라 및 대여소 데이터 로드
station_loc_df  = ('station_loc_df', 'station_loc')
raw_park     = ('infra_park_df', 'infra_park')
raw_river    = ('infra_river_df', 'infra_river')
raw_subway   = ('infra_subway_df', 'infra_subway')
raw_business = ('infra_business_df', 'infra_business')
raw_univ     = ('infra_univ_df', 'infra_univ')
raw_school   = ('infra_school_df', 'infra_school')

if len(station_loc_df) == 0:
    print("\n[경고] 따릉이 대여소(station_loc) 데이터가 비어있습니다!")


# ==========================================
# 2. 결측치 제거 및 공간 데이터(GeoDataFrame) 변환
# ==========================================
print("\n========== [2단계] 공간 데이터(GeoDataFrame) 변환 시작 ==========")

def get_clean_gdf(df, wkt_col=None):
    """자동 컬럼 감지를 통한 GeoDataFrame 변환 전처리 함수"""
    if df.empty:
        return gpd.GeoDataFrame()

    # 하천 데이터(WKT) 처리
    if wkt_col:
        df = df.dropna(subset=[wkt_col]).copy()
        df['geometry'] = df[wkt_col].apply(
            lambda x: wkt.loads(str(x)) if pd.notna(x) and str(x) != 'None' else None
        )
        df = df.dropna(subset=['geometry'])
        gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

    # 점 데이터 (위경도 자동 감지) 처리
    else:
        lat_col = 'latitude' if 'latitude' in df.columns else 'lat' if 'lat' in df.columns else None
        lon_col = 'longitude' if 'longitude' in df.columns else 'lon' if 'lon' in df.columns else 'lot' if 'lot' in df.columns else None

        if not lat_col or not lon_col:
            print("[경고] 위경도 컬럼을 찾을 수 없습니다! 카카오 API 좌표 변환을 먼저 실행했는지 확인하세요.")
            return gpd.GeoDataFrame()

        df = df.dropna(subset=[lat_col, lon_col]).copy()
        gdf = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df[lon_col].astype(float), df[lat_col].astype(float)),
            crs="EPSG:4326"
        )

    # 좌표계 변환(미터 단위 측정용) 및 빈 공간 데이터 제거
    gdf = gdf.to_crs(epsg=5179)
    gdf = gdf[~gdf.is_empty]

    return gdf

# ------------------------------------------
# 대여소 및 인프라 데이터 전처리 적용
# ------------------------------------------
station_loc_df['lat'] = pd.to_numeric(station_loc_df.get('lat'), errors='coerce')
station_loc_df['lon'] = pd.to_numeric(station_loc_df.get('lon'), errors='coerce')
gdf_station = get_clean_gdf(station_loc_df)
print(f"[완료] 대여소(Station) 최종: {len(gdf_station)}건 변환 완료")

# 공원 데이터: 면적 숫자 강제 추출 및 좌표 매핑
raw_park['park_area'] = raw_park['area'].apply(lambda x: float(re.sub(r'[^0-9.]', '', str(x))) if pd.notna(x) else np.nan)
raw_park['lon'] = raw_park['xcrd_g'].fillna(raw_park['xcrd'])
raw_park['lat'] = raw_park['ycrd_g'].fillna(raw_park['ycrd'])

# 지하철 데이터: 좌표 숫자 강제 변환
if 'lat' in raw_subway.columns:
    raw_subway['lat'] = pd.to_numeric(raw_subway['lat'], errors='coerce')
if 'lot' in raw_subway.columns:
    raw_subway['lot'] = pd.to_numeric(raw_subway['lot'], errors='coerce')
elif 'lon' in raw_subway.columns:
    raw_subway['lon'] = pd.to_numeric(raw_subway['lon'], errors='coerce')

# GeoDataFrame 일괄 생성
gdf_park     = get_clean_gdf(raw_park)
gdf_river    = get_clean_gdf(raw_river, wkt_col='geom_wkt')
gdf_subway   = get_clean_gdf(raw_subway)
gdf_business = get_clean_gdf(raw_business)
gdf_univ     = get_clean_gdf(raw_univ)
gdf_school   = get_clean_gdf(raw_school)

# 500m 반경 카운트를 위해 교육 시설(학교+대학교) 통합
gdf_edu = pd.concat([gdf_school, gdf_univ], ignore_index=True)


# ==========================================
# 3. 인프라 반경 및 최단거리 계산 (Feature Engineering)
# ==========================================
print("\n========== [3단계] 대여소 기준 인프라 피처(Feature) 연산 시작 ==========")

if not gdf_station.empty:
    master_df = gdf_station.copy()

    # [내부 함수 1] 지정된 반경 내 대상 인프라 개수 계산
    def get_count_in_buffer(target_gdf, source_gdf, radius_m, col_name):
        if source_gdf.empty:
            return pd.Series(0, index=target_gdf.index, name=col_name)

        buffer_gdf = target_gdf.copy()
        buffer_gdf['geometry'] = buffer_gdf.geometry.buffer(radius_m)
        joined = gpd.sjoin(buffer_gdf, source_gdf, how='left', predicate='intersects')
        count = joined.groupby(joined.index)['index_right'].count()

        return count.rename(col_name)

    # [내부 함수 2] 대상 인프라까지의 최단 거리 계산
    def get_nearest_dist(target_gdf, source_gdf, col_name):
        if source_gdf.empty:
            return pd.DataFrame({col_name: [np.nan]*len(target_gdf)}, index=target_gdf.index)

        nearest = gpd.sjoin_nearest(target_gdf, source_gdf, distance_col=col_name)
        nearest = nearest[~nearest.index.duplicated(keep='first')]

        return nearest[[col_name]]

    # ------------------------------------------
    # 피처 1. 반경 내 개수 계산
    # ------------------------------------------
    print("피처 연산 중 (1/2): 반경 내 인프라 카운트 (약 10~30초 소요)...")
    master_df['subway_cnt_300m'] = get_count_in_buffer(master_df, gdf_subway, 300, 'subway_cnt_300m')
    master_df['biz_cnt_300m']    = get_count_in_buffer(master_df, gdf_business, 300, 'biz_cnt_300m')
    master_df['edu_cnt_500m']    = get_count_in_buffer(master_df, gdf_edu, 500, 'edu_cnt_500m')
    master_df['park_cnt_500m']   = get_count_in_buffer(master_df, gdf_park, 500, 'park_cnt_500m')
    master_df['river_cnt_1km']   = get_count_in_buffer(master_df, gdf_river, 1000, 'river_cnt_1km')

    # ------------------------------------------
    # 피처 2. 최단 거리 계산
    # ------------------------------------------
    print("피처 연산 중 (2/2): 최단 거리 계산...")
    master_df = master_df.join(get_nearest_dist(master_df, gdf_subway, 'dist_subway'))
    master_df = master_df.join(get_nearest_dist(master_df, gdf_river, 'dist_river'))

    # ------------------------------------------
    # 최종 데이터프레임 정리 (컬럼 재배치)
    # ------------------------------------------
    master_df = master_df.drop(columns=['geometry'])
    base_cols = ['station_id', 'address_1', 'lat', 'lon', 'district']

    # 에러 방지를 위해 실제 존재하는 base_cols만 추출
    actual_base_cols = [c for c in base_cols if c in master_df.columns]
    feature_cols = ['subway_cnt_300m', 'biz_cnt_300m', 'edu_cnt_500m', 'park_cnt_500m', 'river_cnt_1km', 'dist_subway', 'dist_river']
    other_cols = [c for c in master_df.columns if c not in actual_base_cols + feature_cols]

    master_df = master_df[actual_base_cols + feature_cols + other_cols]

    print(f"\n========== [완료] 최종 마스터 데이터 구조 완성 (총 {len(master_df)}건) ==========")
    display(HTML("<h3>대여소 기준 인프라 마스터 데이터 (상위 5개)</h3>"))
    display(master_df.head(5))

else:
    print("\n========== [경고] 대여소 데이터가 비어 있어 작업을 중단합니다 ==========")

## EDA

## 피처(X) / 타깃(Y) 분리

## Train / Validation / Test 3분할

## 평가 지표 함수 및 기본 모델 학습

## 여러 모델 비교(Ridge, RandomForest, XGBoost, LightGBM)

## 앙상블(Voting Regressor)

## 최종 모델 선택

## Test셋 최종 평가

## 모델 저장(pkl) 및 저장된 모델 검증